# USA Regional Sales — analysis of the committed processed export

Tutorial-based learning project. The original multi-sheet workbook is not supplied. This notebook starts from the existing processed CSV; it does not claim to reproduce the unavailable raw joins. Amounts are USD. Cost is interpreted as unit cost, matching the original transformation.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("Sales_data(EDA Exported).csv", parse_dates=["order_date"])
required={"revenue","quantity","cost","order_number","product_name","us_region","channel"}
assert required <= set(df.columns)
assert df[["revenue","quantity","cost"]].notna().all().all()
df["total_cost"] = df["quantity"] * df["cost"]
df["profit"] = df["revenue"] - df["total_cost"]
print("Rows:", len(df), "Revenue USD:", df["revenue"].sum(), "Profit USD:", df["profit"].sum())

Rows: 64104 Revenue USD: 1235968899.0 Profit USD: 461772559.92


In [2]:
monthly=df.groupby(df["order_date"].dt.to_period("M"))["revenue"].sum()
monthly.plot(figsize=(12,4),title="Monthly revenue (USD)",ylabel="USD")
plt.tight_layout(); plt.show()

In [3]:
for column in ["product_name","us_region","channel"]:
    totals=df.groupby(column)[["revenue","profit"]].sum().sort_values("revenue",ascending=False)
    print(column, totals.head(10))
    totals.head(10).plot.bar(title=f"Revenue and profit by {column} (USD)")
    plt.tight_layout(); plt.show()

product_name                   revenue        profit
product_name                           
Product 26    117291821.4  4.440670e+07
Product 25    109473966.6  4.161245e+07
Product 13     78281379.6  2.871081e+07
Product 14     75390396.6  2.692773e+07
Product 5      70804380.6  2.658405e+07
Product 15     67331623.2  2.476014e+07
Product 2      57401097.6  2.056627e+07
Product 4      56701537.2  2.134395e+07
Product 1      55952289.6  2.144745e+07
Product 3      51764816.4  1.902420e+07
us_region                revenue        profit
us_region                           
West       372142307.6  1.392671e+08
South      335135011.7  1.251712e+08
Midwest    320322363.6  1.200967e+08
Northeast  208369216.1  7.723753e+07
channel                  revenue        profit
channel                               
Wholesale    668197244.4  2.473479e+08
Distributor  387139788.6  1.457701e+08
Export       180631866.0  6.865458e+07


In [4]:
orders=df.groupby("order_number")["revenue"].sum()
print("Average order value USD:", orders.mean())
orders.plot.hist(bins=40,title="Order value distribution (USD)"); plt.show()
channel=df.groupby("channel")[["revenue","profit"]].sum()
channel["weighted_margin_pct"] = channel["profit"].div(channel["revenue"]).mul(100)
print(channel)

Average order value USD: 115684.09762261325
                 revenue        profit  weighted_margin_pct
channel                                                    
Distributor  387139788.6  1.457701e+08            37.653091
Export       180631866.0  6.865458e+07            38.008010
Wholesale    668197244.4  2.473479e+08            37.017197


## Interpretation boundaries
Revenue/profit concentration and seasonality are descriptive. Dashboard images and presentation assets are retained; no editable Power BI source is supplied. Earlier placeholder claims such as “Product X drove 30%” are not treated as results.